![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 07: Geospatial & Spatial Epidemiology
***
**Learning objectives**
- Understand spatial autocorrelation: Tobler's First Law of Geography
- Build spatial weights matrices (queen contiguity, rook, distance-based, k-NN)
- Compute Global Moran's I and its significance via permutation
- Calculate Local Moran's I (LISA) to identify spatial clusters
- Produce cluster maps: High-High, Low-Low, High-Low, Low-High quadrants
- Interpret Moran scatterplots for health outcomes

**Estimated time:** 3 hours | **Level:** Advanced  
**Libraries:** `numpy`, `scipy`, `matplotlib`, `pysal` (optional)


## 1. Setup & Dataset

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore"); import os; os.makedirs("/tmp/mod07", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)
NROW, NCOL = 10, 20; N = NROW * NCOL
row_idx = np.repeat(np.arange(NROW), NCOL)
col_idx = np.tile(np.arange(NCOL), NROW)
lon = -120 + col_idx*2.5 + np.random.normal(0,0.3,N)
lat =   28 + row_idx*2.0 + np.random.normal(0,0.2,N)
pct_poverty   = 10 + 8*np.random.beta(2,5,N) + 3*np.sin(lon/20)
pct_uninsured = 8  + 6*np.random.beta(2,4,N) + 0.3*pct_poverty
pm25          = 8  + 4*np.random.gamma(2,1,N)
population    = np.random.lognormal(10.5,1.0,N).astype(int)
from scipy.ndimage import gaussian_filter
def spatial_smooth(vals, sigma=2):
    grid=np.zeros((NROW,NCOL))
    for r,c,v in zip(row_idx,col_idx,vals): grid[r,c]=v
    return gaussian_filter(grid,sigma=sigma)[row_idx,col_idx]
spatial_risk = spatial_smooth(np.random.normal(0,1,N))
cvd_rate = (180 + 1.2*pct_poverty + 0.8*pct_uninsured + 0.5*pm25
            + 15*spatial_risk + np.random.normal(0,12,N))
expected = (cvd_rate/100000)*population*5
observed = np.random.poisson(expected).astype(int)
smr = observed / expected.clip(1)
df = pd.DataFrame({"county_id":[f"C{i:03d}" for i in range(N)],
    "lon":lon,"lat":lat,"row":row_idx,"col":col_idx,
    "pct_poverty":pct_poverty,"pct_uninsured":pct_uninsured,"pm25":pm25,
    "population":population,"cvd_rate":cvd_rate,
    "observed":observed,"expected":expected,"smr":smr,"spatial_risk":spatial_risk})
print(f"N={N} counties | CVD={cvd_rate.mean():.1f}/100k | Poverty={pct_poverty.mean():.1f}%")

## 2. Spatial Weights Matrix

A **spatial weights matrix W** encodes the neighbourhood structure:
- **Queen contiguity**: shares any boundary point or corner
- **Rook contiguity**: shares an edge only
- **Distance-based**: w_ij = 1 if distance < threshold
- **k-NN**: each unit connected to its k nearest neighbours
- **Inverse distance**: w_ij = 1/d_ij (continuous)

Row-standardisation: W* where each row sums to 1 (so W*x = spatial lag of x)


In [ ]:
from scipy.spatial import cKDTree

def build_queen_weights(row_idx, col_idx, nrow, ncol):
    """Build queen contiguity weights (8 neighbours) on a grid."""
    N = len(row_idx)
    W = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            if i != j:
                dr = abs(row_idx[i] - row_idx[j])
                dc = abs(col_idx[i] - col_idx[j])
                if dr <= 1 and dc <= 1:
                    W[i, j] = 1
    # Row-standardise
    row_sums = W.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return W / row_sums

def build_knn_weights(coords, k=5):
    """Build k-nearest neighbour spatial weights."""
    tree = cKDTree(coords)
    dists, idxs = tree.query(coords, k=k+1)  # k+1 because includes self
    N = len(coords)
    W = np.zeros((N, N))
    for i in range(N):
        for j in idxs[i][1:]:  # skip self (index 0)
            W[i, j] = 1
    row_sums = W.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return W / row_sums

def build_distance_weights(coords, threshold=None, inverse=False):
    """Build distance-based weights with optional inverse weighting."""
    tree = cKDTree(coords)
    if threshold is None:
        threshold = np.percentile(tree.query(coords, k=2)[0][:,1], 90) * 2
    N = len(coords)
    W = np.zeros((N, N))
    pairs = tree.query_pairs(threshold)
    for (i, j) in pairs:
        d = np.linalg.norm(coords[i] - coords[j])
        w = 1/d if inverse and d > 0 else 1.0
        W[i, j] = W[j, i] = w
    row_sums = W.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return W / row_sums

# Build weights on grid structure
coords = np.column_stack([df.lon, df.lat])
W_queen = build_queen_weights(df.row.values, df.col.values, NROW, NCOL)
W_knn   = build_knn_weights(coords, k=6)
W_dist  = build_distance_weights(coords)

print(f"Queen W: {N}x{N} | Mean neighbours: {(W_queen>0).sum(axis=1).mean():.1f}")
print(f"KNN W  : {N}x{N} | Mean neighbours: {(W_knn>0).sum(axis=1).mean():.1f}")
print(f"Dist W : {N}x{N} | Mean neighbours: {(W_dist>0).sum(axis=1).mean():.1f}")


## 3. Global Moran's I

In [ ]:
def morans_i(y, W, n_perm=999, seed=42):
    """
    Compute Global Moran's I with permutation inference.
    I = (N / sum(W)) * (z'Wz) / (z'z)
    where z = y - mean(y)
    """
    n = len(y)
    z = y - y.mean()
    S0 = W.sum()
    Wz = W @ z
    I_obs = (n / S0) * (z @ Wz) / (z @ z)
    # Permutation test
    rng = np.random.default_rng(seed)
    I_perm = np.zeros(n_perm)
    for b in range(n_perm):
        z_perm = rng.permutation(z)
        Wz_p = W @ z_perm
        I_perm[b] = (n / S0) * (z_perm @ Wz_p) / (z_perm @ z_perm)
    p_value = (np.abs(I_perm) >= np.abs(I_obs)).mean()
    e_i = -1 / (n - 1)
    return {"I":I_obs, "E[I]":e_i, "p_value":p_value,
            "I_perm":I_perm, "z_score":(I_obs-e_i)/I_perm.std()}

print("Global Moran's I Results:\n")
print(f"{'Variable':20s} {'Moran I':>10s} {'E[I]':>8s} {'z-score':>10s} {'p-value':>10s}")
print("-"*62)
for var, values in [("CVD rate",df.cvd_rate),("Poverty (%)",df.pct_poverty),
                     ("SMR",df.smr),("PM2.5",df.pm25)]:
    res = morans_i(values.values, W_queen)
    sig = "***" if res["p_value"]<0.001 else "**" if res["p_value"]<0.01 else "*" if res["p_value"]<0.05 else ""
    print(f"  {var:18s} {res['I']:>10.4f} {res['E[I]']:>8.4f} {res['z_score']:>10.2f} {res['p_value']:>8.4f} {sig}")

# Moran I for CVD rate (main analysis)
res_cvd = morans_i(df.cvd_rate.values, W_queen)


In [ ]:
# Moran scatterplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Moran scatterplot: y vs spatial lag (W*y)
z_cvd = (df.cvd_rate - df.cvd_rate.mean()) / df.cvd_rate.std()
Wz_cvd = W_queen @ z_cvd
quadrant = ((z_cvd > 0) & (Wz_cvd > 0)).astype(int)*1 +            ((z_cvd < 0) & (Wz_cvd < 0)).astype(int)*2 +            ((z_cvd > 0) & (Wz_cvd < 0)).astype(int)*3 +            ((z_cvd < 0) & (Wz_cvd > 0)).astype(int)*4

quad_colors = {1:"#D65F5F",2:"#4878CF",3:"#AEC6CF",4:"#F4A582"}
quad_labels = {1:"High-High",2:"Low-Low",3:"High-Low",4:"Low-High"}
colors_moran = [quad_colors.get(q,"gray") for q in quadrant]

axes[0].scatter(z_cvd, Wz_cvd, c=colors_moran, alpha=0.7, s=30, edgecolors="none")
m_slope = np.polyfit(z_cvd, Wz_cvd, 1)[0]
xl = np.linspace(z_cvd.min(), z_cvd.max(), 100)
axes[0].plot(xl, m_slope*xl, "k-", lw=2, label=f"Slope = Moran's I = {res_cvd['I']:.3f}")
axes[0].axhline(0, color="gray", lw=0.8); axes[0].axvline(0, color="gray", lw=0.8)
axes[0].set_xlabel("Standardised CVD rate (z)"); axes[0].set_ylabel("Spatial lag (W*z)")
axes[0].set_title(f"Moran Scatterplot: CVD Rate\n(I={res_cvd['I']:.3f}, p={res_cvd['p_value']:.4f})")
axes[0].legend(fontsize=8)
for q in [1,2,3,4]:
    axes[0].plot([],[],"o",color=quad_colors[q],label=quad_labels[q],ms=8)
axes[0].legend(fontsize=7)

# Permutation distribution
axes[1].hist(res_cvd["I_perm"], bins=40, color="#AEC6CF", edgecolor="white", alpha=0.9)
axes[1].axvline(res_cvd["I"], color="#D65F5F", lw=2.5, label=f"Observed I={res_cvd['I']:.4f}")
axes[1].axvline(res_cvd["E[I]"], color="gray", ls="--", lw=1.5, label=f"E[I]={res_cvd['E[I]']:.4f}")
axes[1].set_xlabel("Moran's I (permuted)"); axes[1].set_ylabel("Count")
axes[1].set_title(f"Permutation Distribution (B=999)\np={res_cvd['p_value']:.4f}")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig("/tmp/mod07/morans_i.png", bbox_inches="tight"); plt.show()


## 4. Local Moran's I (LISA)

In [ ]:
def local_morans_i(y, W, n_perm=499, seed=42):
    """Compute Local Indicators of Spatial Association (LISA)."""
    n = len(y)
    z = y - y.mean()
    var_z = z.var()
    # Local I_i = z_i * sum_j(w_ij * z_j) / var(z)
    Wz = W @ z
    I_local = z * Wz / var_z
    # Pseudo p-values via conditional permutation
    rng = np.random.default_rng(seed)
    I_perm = np.zeros((n_perm, n))
    for b in range(n_perm):
        z_p = rng.permutation(z)
        Wz_p = W @ z_p
        I_perm[b] = z_p * Wz_p / var_z
    p_values = (np.abs(I_perm) >= np.abs(I_local[np.newaxis,:])).mean(axis=0)
    # Classify quadrants
    z_lag = Wz / z.std()
    z_std = z / z.std()
    quadrant = np.where((z_std>0)&(z_lag>0), "HH",
               np.where((z_std<0)&(z_lag<0), "LL",
               np.where((z_std>0)&(z_lag<0), "HL", "LH")))
    cluster = np.where(p_values<0.05, quadrant, "NS")
    return {"I_local":I_local,"p_values":p_values,"cluster":cluster}

lisa_res = local_morans_i(df.cvd_rate.values, W_queen)
df["lisa_cluster"] = lisa_res["cluster"]
df["lisa_i"]       = lisa_res["I_local"]
df["lisa_p"]       = lisa_res["p_values"]

# Cluster map
cluster_colors = {"HH":"#D65F5F","LL":"#4878CF","HL":"#F4A582","LH":"#92C5DE","NS":"#E0E0E0"}
colors_lisa = [cluster_colors[c] for c in df.lisa_cluster]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].scatter(df.lon, df.lat, c=colors_lisa, s=200, edgecolors="white", linewidth=0.3)
axes[0].set_title("LISA Cluster Map: CVD Mortality\n(HH=hot spots, LL=cold spots, HL/LH=outliers)", fontsize=10)
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
for label, color in cluster_colors.items():
    n_cl = (df.lisa_cluster==label).sum()
    axes[0].plot([],[],"o",color=color,ms=10,label=f"{label} (n={n_cl})")
axes[0].legend(fontsize=9)

# Significance map
sig_colors = {True:"#D65F5F", False:"#E0E0E0"}
sig_mask = lisa_res["p_values"] < 0.05
axes[1].scatter(df.lon, df.lat, c=[sig_colors[s] for s in sig_mask],
                s=200, edgecolors="white", linewidth=0.3)
axes[1].set_title(f"LISA Significance Map (p<0.05)\n{sig_mask.sum()} significant counties ({sig_mask.mean()*100:.1f}%)")
axes[1].set_xlabel("Longitude"); axes[1].set_ylabel("Latitude")
plt.tight_layout()
plt.savefig("/tmp/mod07/lisa_cluster_map.png", bbox_inches="tight"); plt.show()

# Summary
print("LISA cluster summary:")
for label in ["HH","LL","HL","LH","NS"]:
    n = (df.lisa_cluster==label).sum()
    print(f"  {label}: {n:4d} counties ({n/len(df)*100:.1f}%)")


## 5. Getis-Ord G* Hot Spot Analysis

In [ ]:
def getis_ord_gstar(y, W_binary, W_row_std=None):
    """
    Getis-Ord G* statistic for hot spot analysis.
    G*_i = (sum_j w_ij*x_j - X_bar*sum_j w_ij) /
            S * sqrt((n*sum_j w_ij^2 - (sum_j w_ij)^2) / (n-1))
    """
    n = len(y)
    X_bar = y.mean()
    S = y.std()
    G_star = np.zeros(n)
    for i in range(n):
        w_i = W_binary[i]
        sum_w = w_i.sum()
        sum_wx = (w_i * y).sum()
        num = sum_wx - X_bar * sum_w
        sum_w2 = (w_i**2).sum()
        denom_sq = sum_w2 - sum_w**2/n
        if denom_sq > 0 and S > 0:
            denom = S * np.sqrt((n*sum_w2 - sum_w**2) / (n-1))
            G_star[i] = num / denom if denom > 0 else 0
        else:
            G_star[i] = 0
    return G_star

# Build binary (non-standardised) queen weights
W_binary = (W_queen > 0).astype(float)
np.fill_diagonal(W_binary, 1)  # G* includes self

g_star = getis_ord_gstar(df.cvd_rate.values, W_binary)
from scipy.stats import norm as sp_norm
p_gstar = 2 * sp_norm.sf(np.abs(g_star))

df["g_star"]   = g_star
df["g_star_p"] = p_gstar

fig, ax = plt.subplots(figsize=(10, 6))
hotspot_class = np.where((g_star > 2.58) & (p_gstar<0.01), "Hot spot 99%",
                np.where((g_star > 1.96) & (p_gstar<0.05), "Hot spot 95%",
                np.where((g_star < -2.58) & (p_gstar<0.01), "Cold spot 99%",
                np.where((g_star < -1.96) & (p_gstar<0.05), "Cold spot 95%", "Not significant"))))

hs_colors = {"Hot spot 99%":"#D73027","Hot spot 95%":"#FC8D59",
             "Cold spot 99%":"#4575B4","Cold spot 95%":"#91BFDB","Not significant":"#E0E0E0"}
colors_hs = [hs_colors[h] for h in hotspot_class]
ax.scatter(df.lon, df.lat, c=colors_hs, s=200, edgecolors="white", linewidth=0.3)
ax.set_title("Getis-Ord G* Hot Spot Analysis: CVD Mortality", fontsize=11)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
for label, color in hs_colors.items():
    n_hs = np.sum(hotspot_class==label)
    ax.plot([],[],"o",color=color,ms=10,label=f"{label} (n={n_hs})")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("/tmp/mod07/getis_ord_gstar.png", bbox_inches="tight"); plt.show()


## 6. Knowledge Check
1. Global Moran's I = 0.42 (p<0.001). Does this tell us *where* the spatial clusters are?
2. A LISA map shows an HL outlier (High-Low): a high CVD county surrounded by low-CVD counties. List three explanations.
3. You used queen contiguity weights. How would switching to k-NN (k=8) change the Moran's I estimate?
4. Why should LISA p-values be interpreted cautiously given that you're running 200 local tests?
5. Compute Moran's I for the *residuals* from an OLS regression of CVD on poverty. Interpret the result.


In [ ]:
# Exercise 5 — Moran's I on OLS residuals
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(df[["pct_poverty","pct_uninsured","pm25"]], df["cvd_rate"])
residuals = df["cvd_rate"] - lr.predict(df[["pct_poverty","pct_uninsured","pm25"]])
res_ols = morans_i(residuals.values, W_queen, n_perm=499)
print(f"Moran's I on CVD rate           : {res_cvd['I']:.4f} (p={res_cvd['p_value']:.4f})")
print(f"Moran's I on OLS residuals      : {res_ols['I']:.4f} (p={res_ols['p_value']:.4f})")
print()
if res_ols["p_value"] < 0.05:
    print("Significant residual spatial autocorrelation -> OLS is biased!")
    print("Need spatial regression models (spatial lag or spatial error).")
else:
    print("Residual spatial autocorrelation removed by covariate adjustment.")


***
**Next → NB-03: Spatial Regression — Lag, Error & GWR**
